# SRD knowledge graph

Builds the graph the chatbot retrieves from and draws its nodes and edges.

- Nodes are D&D entities (Spell, Monster, MagicItem, Class, Condition, ...) and the plain sections around them. Each node's `label` is its entity type.
- Edges are typed relationships such as `CAST_BY`, `DEALS_DAMAGE`, `IMMUNE_TO_CONDITION` and `GRANTS_SPELL`. `CONTAINS` is the header hierarchy and `REFERENCES` is a section naming another section with no more specific meaning.

How the graph is built is described at the top of `src/graphrag/graph.py`.

The interactive views are written to `notebooks/output/` as HTML files. Open them in a browser if your notebook viewer doesn't render iframes.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import HTML
from IPython.display import display
from pyvis.network import Network

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.graphrag.graph import CONTAINS  # noqa: E402
from src.graphrag.graph import REFERENCES  # noqa: E402
from src.graphrag.graph import build_graph  # noqa: E402

OUTPUT_DIR = REPO_ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

graph = build_graph(REPO_ROOT / "srd_extraction.md")
print(f"{graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges")

2,358 nodes, 15,771 edges


## Nodes and relationships

Node counts per entity label, then every relationship in the graph: which label it starts from, its type, which label it points to, and how many edges there are.

In [2]:
nodes = pd.DataFrame.from_dict(dict(graph.nodes(data=True)), orient="index")
edges = pd.DataFrame(
    [
        {
            "source": graph.nodes[source]["label"],
            "target": graph.nodes[target]["label"],
            "type": data["type"],
        }
        for source, target, data in graph.edges(data=True)
    ]
)

display(nodes["label"].value_counts().rename("nodes").to_frame().T)

relationships = edges.value_counts(["source", "type", "target"]).rename("edges").reset_index()
with pd.option_context("display.max_rows", None):
    display(relationships)

label,Section,Spell,Monster,MagicItem,Feature,Tool,Skill,Feat,Condition,CreatureType,...,Class,Subclass,Action,Race,Alignment,SchoolOfMagic,AbilityScore,Rarity,Size,Background
nodes,1013,339,330,258,232,25,18,17,15,14,...,12,12,12,9,9,8,6,6,6,4


,source,type,target,edges
0,Monster,REFERENCES,Section,2573
1,Spell,REFERENCES,Section,1601
2,Section,REFERENCES,Spell,996
3,Section,CONTAINS,Section,968
4,Section,REFERENCES,Section,940
5,Spell,CAST_BY,Class,879
6,Monster,CAN_CAST,Spell,430
7,Monster,PROFICIENT_IN,Skill,429
8,Monster,HAS_SIZE,Size,373
9,Monster,IMMUNE_TO_CONDITION,Condition,371


In [3]:
# How many edges connect each pair of labels, whatever their type.
pd.crosstab(edges["source"], edges["target"])

target,AbilityScore,Action,Alignment,Background,Class,Condition,CreatureType,DamageType,Feat,Feature,...,Monster,Race,Rarity,SchoolOfMagic,Section,Size,Skill,Spell,Subclass,Tool
source,,,,,,,,,,,,,,,,,,,,,
Action,0,1,0,0,0,2,0,0,0,0,...,0,0,0,0,18,0,0,0,0,0
Background,0,0,0,0,2,0,0,0,4,0,...,0,0,0,0,26,0,8,0,0,4
Class,16,0,0,0,0,0,0,0,2,0,...,0,0,0,0,142,0,0,11,12,4
Condition,0,7,0,0,0,6,0,0,0,0,...,0,0,0,0,29,0,0,2,0,0
Feat,0,3,0,0,2,3,0,0,0,0,...,0,0,0,0,32,0,0,0,0,0
Feature,0,41,0,0,84,45,0,0,32,0,...,4,0,0,0,275,0,0,115,0,3
MagicItem,0,106,0,1,36,73,0,11,1,0,...,96,1,263,0,366,0,0,211,0,0
Monster,0,85,205,0,6,604,324,351,0,0,...,12,0,0,0,2573,373,429,430,0,1
Race,0,4,0,0,1,6,0,0,1,0,...,6,0,0,0,42,11,0,32,0,0


In [4]:
# The most linked-to nodes.
in_degree = pd.Series(dict(graph.in_degree()), name="incoming edges")
nodes.join(in_degree).nlargest(15, "incoming edges")[
    ["label", "breadcrumb", "incoming edges"]
]

,label,breadcrumb,incoming edges
752,Section,Spells > Casting Spells > Casting Time,341
757,Section,Spells > Casting Spells > Components,341
1229,Section,Rules Glossary > Rules Definitions > Passive P...,340
1675,Section,Monsters > Parts of a Stat Block > Languages,340
1134,Section,Rules Glossary > Rules Definitions > Attack Roll,338
1125,Section,Rules Glossary > Rules Definitions > Action,297
479,Class,Classes > Wizard,256
1680,Section,Monsters > Parts of a Stat Block > Traits,247
1668,Section,Monsters > Parts of a Stat Block > Skills,220
1220,Action,Rules Glossary > Rules Definitions > Magic [Ac...,206


## Full graph

Every node and edge. Node color is the entity label, node size grows with the number of connections. Dark edges are `CONTAINS`, blue edges are the typed D&D relationships and faint gray edges are `REFERENCES`. Hover a node for its label and breadcrumb and an edge for its type. Scroll to zoom, drag to pan.

The layout is computed once with networkx and physics is turned off, so the browser stays responsive with all edges drawn. Edges are hidden while you drag.

In [5]:
# The seven most common entity labels get a color each, the rest share gray.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
OTHER_COLOR = "#8a8a85"
TYPED = "typed relationship"
EDGE_COLORS = {CONTAINS: "#3d3d3a", TYPED: "#2a78d6", REFERENCES: "#b5b5b0"}
# Thousands of overlapping edges need transparency in the full graph.
FAINT_EDGE_COLORS = {
    CONTAINS: "#3d3d3a",
    TYPED: "rgba(42, 120, 214, 0.3)",
    REFERENCES: "rgba(120, 120, 115, 0.1)",
}

labels = nodes.loc[nodes["label"] != "Section", "label"]

label_colors = dict(
    zip([label for label, _ in Counter(labels).most_common(len(PALETTE))], PALETTE)
)


def node_color(label: str) -> str:
    return label_colors.get(label, OTHER_COLOR)


def edge_kind(edge_type: str) -> str:
    return edge_type if edge_type in (CONTAINS, REFERENCES) else TYPED


def legend() -> HTML:
    swatch = (
        '<span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:{}"></span> {}'
    )
    line = '<span style="display:inline-block;width:18px;border-top:3px solid {}"></span> {}'
    items = [swatch.format(color, name) for name, color in label_colors.items()]
    items.append(swatch.format(OTHER_COLOR, "Section and other labels"))
    items += [line.format(color, edge_type) for edge_type, color in EDGE_COLORS.items()]
    return HTML(
        '<div style="display:flex;flex-wrap:wrap;gap:6px 18px">'
        + "".join(f'<span style="white-space:nowrap">{item}</span>' for item in items)
        + "</div>"
    )


def draw(subgraph: nx.DiGraph, filename: str, positions: dict | None = None) -> HTML:
    """Write a subgraph to an interactive HTML file and return an iframe showing it.

    With `positions` the nodes are pinned and physics is off. Without, vis.js lays the
    graph out and edges are labeled with their type.
    """
    net = Network(height="750px", width="100%", directed=True, cdn_resources="in_line")
    for node, data in subgraph.nodes(data=True):
        options = {}
        if positions is not None:
            x, y = positions[node]
            options = {"x": float(x) * 4000, "y": float(y) * 4000, "physics": False}
        net.add_node(
            node,
            label=data["header"] if positions is None else " ",
            title=f"{data['label']}: {data['breadcrumb']}",
            color=node_color(data["label"]),
            size=8 + 2 * subgraph.degree(node) ** 0.5,
            **options,
        )
    for source, target, data in subgraph.edges(data=True):
        kind = edge_kind(data["type"])
        options = {"color": FAINT_EDGE_COLORS[kind]}
        if positions is None:
            options = {
                "color": EDGE_COLORS[kind],
                "label": data["type"],
                "font": {"size": 9, "color": "#6b6b66", "strokeWidth": 3},
            }
        net.add_edge(source, target, title=data["type"], width=1, **options)
    if positions is not None:
        net.set_options(
            '{"physics": {"enabled": false}, "edges": {"smooth": false},'
            ' "interaction": {"hideEdgesOnDrag": true}}'
        )

    path = OUTPUT_DIR / filename
    path.write_text(net.generate_html(notebook=False), encoding="utf-8")
    relative = path.relative_to(Path.cwd()) if path.is_relative_to(Path.cwd()) else path
    return HTML(
        f'<iframe src="{relative.as_posix()}" width="100%" height="770"'
        ' style="border:1px solid #ccc"></iframe>'
    )

In [6]:
# Pulling harder along CONTAINS edges keeps each chapter's nodes together.
layout_graph = graph.to_undirected()
for _, _, data in layout_graph.edges(data=True):
    data["weight"] = 5 if data["type"] == CONTAINS else 0.2
positions = nx.spring_layout(layout_graph, seed=7, iterations=100, weight="weight")

display(legend())
draw(graph, "srd_graph.html", positions)

/tmp/claude-0/-home-user/09548ae6-6723-54db-a98e-1d7590b802d6/scratchpad/venv/lib/python3.12/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Neighborhood of one section

The full graph shows the overall shape. To read individual relationships, draw one section with everything it links to and everything that links to it. Edges are labeled with their type. `draw_neighborhood` takes the header of any entity.

In [7]:
def draw_neighborhood(header: str, radius: int = 1) -> HTML:
    """Draw the sections within `radius` hops of the first section with this header."""
    center = next(
        node
        for node, data in graph.nodes(data=True)
        if data["header"] == header and data["label"] != "Section"
    )
    neighborhood = nx.ego_graph(graph, center, radius=radius, undirected=True)
    print(f"{neighborhood.number_of_nodes()} nodes, {neighborhood.number_of_edges()} edges")
    return draw(neighborhood, f"neighborhood_{header.lower().replace(' ', '_')}.html")


display(legend())
draw_neighborhood("Fireball")

31 nodes, 50 edges


In [8]:
display(legend())
draw_neighborhood("Ghoul")

18 nodes, 17 edges
